# Lung internal val — UNI2 A2 best model

Xenium lung, 5 fold-0 slides (all drawn): `VUILD107MA`, `VUHD069`, `TILD175MA`, `TILD117MA1`, `VUHD113`.
Individual-dataset notebook: `code/Xenium_lung/Data_process_visual_xenium_all.ipynb`.

The ROI cells further down stay on `VUILD107MA` (Extended Data Fig. 7a), the largest of these five.

Checkpoint: `data/PanCancer/result_uni2_A2_adjacent_spatial_k8mean_paired_v2_seed42/cross_dataset_cv/A2_adjacent_spatial_k8mean/best_mlp_gpu.pt`  
This is the UNI2 A2 best model. That file is the fold-1 checkpoint, so these notebooks show **fold 0** instead of fold 1. Deployment skipped the 4×3 panel PNGs, so GT versus prediction is drawn from `matched_pred.h5ad`.

```bash
bash code/Internal_val/demo.sh status
ONLY=Lung bash code/Internal_val/demo.sh predict
```


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path("/home/lingyu/ssd2/Python/Hist2Pheno")
if not (REPO / "code").is_dir():
    REPO = Path("/nobackup2/users/lingyu/Python/Hist2Pheno")
sys.path.insert(0, str(REPO / "code" / "Internal_val"))

import internal_val as iv

DATASET = "Lung"
SAVE_FIGURES = True
fold = iv.fold0_table(DATASET)
viz = iv.viz_samples(DATASET)
cov = iv.coverage(DATASET)
print(f"{DATASET} fold {iv.FOLD}: {len(fold)} sections; drawing {len(viz)}")
display(cov)
met = iv.metrics_fold0(DATASET)
if not met.empty:
    cols = [c for c in ("sample", "n_matched", "macro_auc_L1_cells", "macro_auc_L12_cells", "macro_auc_L2_cells") if c in met]
    display(met[cols])
print("viz", viz)


In [ ]:
display(Markdown("## VUILD107MA: cell-type maps (300 dpi PNG) and ROC (PDF). Largest fold-0 lung section."))
sample = "VUILD107MA"
saved = iv.save_celltype_and_roc(
    DATASET,
    sample,
    celltype_figsize=(12.5, 8.8),   # inches: GT + Hist2Pheno, same matched-nuclei canvas
    stardist_figsize=(12.5, 5.5),   # inches: StarDist-all nuclei, its own canvas
    roc_figsize=(1.5, 1.5),         # inches: figure width, side of the square ROC
    bar_figsize=None,               # None keeps height proportional to the class count
    point_s=0.45,                   # scatter area; larger dots on the cell-type maps
    he_image=(
        REPO
        / "data/Xemium/weiqin/SpatialPF-NGenetics/Spatial-PF-Processed/Data/Complete_Cases"
        / sample
        / f"{sample}-HE.tif"
    ),
    he_alpha=0.8,                   # H&E background opacity; lower fades the slide toward white
)
display(Image(filename=str(saved["celltype"])))
if saved["stardist"] is not None:
    display(Image(filename=str(saved["stardist"])))


In [ ]:
display(Markdown("## Existing 4×3 panels (GT / Hist2Pheno / ROC / StarDist-all)"))
missing = []
for sample in viz:
    panel = iv.pred_paths(DATASET, sample)["panel"]
    if panel.is_file():
        display(Markdown(f"### {sample}"))
        display(Image(filename=str(panel)))
    else:
        missing.append(sample)
        print("missing panel", sample)
if missing:
    print("UNI2 A2 deployment did not save 4x3 panel PNGs. GT versus prediction is drawn from matched_pred.h5ad.")


In [ ]:
# display(Markdown("## GT vs pretrained L2 (matched nuclei)"))
# for sample in viz:
#     pred = iv.pred_paths(DATASET, sample)["pred"]
#     if not pred.is_file():
#         print("skip", sample, "(no matched_pred.h5ad)")
#         continue
#     fig = iv.plot_spatial_gt_pred(DATASET, sample, "L2", save=SAVE_FIGURES)
#     display(fig)
#     plt.close(fig)
#     fig = iv.plot_composition(DATASET, sample, "L2", save=SAVE_FIGURES)
#     display(fig)
#     plt.close(fig)


## VUILD107MA H&E ROIs from Extended Data Fig. 7

[Vannan et al., Nature Genetics, Extended Data Fig. 7](https://www.nature.com/articles/s41588-025-02080-x/figures/13). Panel **a** is this section. Panel **d** is six zoomed annotations, and the caption assigns each crop to a different sample:

| Panel d | Sample in the caption |
| --- | --- |
| Granuloma | VUILD96MA |
| TLS | VUILD110LA |
| Microscopic honeycombing | VUILD78MA |
| Goblet cell metaplasia | VUILD104MA2 |
| Hyperplastic AECs | VUILD107MA |
| Fibroblastic focus | VUILD104MA1 |

Only Hyperplastic AECs is taken from this H&E. The cell table has four instances (`hyperplastic_aec_1` to `_4`) with `X_pix_HE` / `Y_pix_HE` on `VUILD107MA-HE.tif`. The other five published crops are not this file, so they are not plotted. Goblet cell metaplasia and fibroblastic focus do occur on this section, but the photographs in panel d are the other samples named above.

图 d 的六张放大图来自六张不同的切片。对照 Extended Data Fig. 7 的图注，只有 Hyperplastic AECs 取自 VUILD107MA，也就是这张 VUILD107MA-HE.tif。其余五张不在这张图上，所以没有裁：

这张切片的细胞表里有 4 个 Hyperplastic AEC 区域（hyperplastic_aec_1 到 _4），坐标是 H&E 像素 X_pix_HE、Y_pix_HE。每个框是该区域全部细胞再向外扩 40 µm。图 a 就是这张全切片；发表的图 d 是其中一处更小的放大，作者没有给出是哪一个 instance。

发表图里的 TLS 仍是 VUILD110LA。这张切片没有 TLS 多边形，但气道上皮下有一团以 B 细胞为主、伴有 T 细胞的淋巴聚集，见后面的 TLS ROI。


In [ ]:
import numpy as np
import tifffile
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from PIL import Image
from shapely import MultiPoint, concave_hull

# The fold-0 loop above leaves sample as the last slide (VUHD113).
sample = "VUILD107MA"
CASE = (
    REPO
    / "data/Xemium/weiqin/SpatialPF-NGenetics/Spatial-PF-Processed/Data/Complete_Cases"
    / sample
)
HE_TIF = CASE / f"{sample}-HE.tif"
HE_LOW = CASE / f"{sample}-HE_lowres.jpg"
CELLS = CASE / f"{sample}_cells_partitioned_by_annotation_sample_match_with_pixel.csv"
ROI_DIR = REPO / "code/Internal_val/figures_vuild107ma"
ROI_DIR.mkdir(parents=True, exist_ok=True)

# VUILD107MA-HE.tif XResolution = 4705882/1000000 px per micron.
PX_PER_UM = 4705882 / 1_000_000
PAD = int(round(40 * PX_PER_UM))

cells = pd.read_csv(
    CELLS, usecols=["Annotation_Type", "annotation_type_instance", "X_pix_HE", "Y_pix_HE"]
)
haec = cells.loc[cells["Annotation_Type"] == "Hyperplastic AECs"]
he = tifffile.memmap(HE_TIF)
H, W = he.shape[:2]
low = np.asarray(Image.open(HE_LOW))
sx, sy = low.shape[1] / W, low.shape[0] / H

boxes = []
for name, g in haec.groupby("annotation_type_instance"):
    x0 = max(0, int(np.floor(g["X_pix_HE"].min()) - PAD))
    x1 = min(W, int(np.ceil(g["X_pix_HE"].max()) + PAD))
    y0 = max(0, int(np.floor(g["Y_pix_HE"].min()) - PAD))
    y1 = min(H, int(np.ceil(g["Y_pix_HE"].max()) + PAD))
    boxes.append((str(name), x0, x1, y0, y1, len(g)))

colors = ["#E64B35", "#4DBBD5", "#00A087", "#3C5488"]
fig, ax = plt.subplots(
    figsize=(low.shape[1] / 300, low.shape[0] / 300), dpi=300, facecolor="white"
)
ax.imshow(low)
handles = []
for (name, x0, x1, y0, y1, n), color in zip(boxes, colors):
    ax.add_patch(
        Rectangle((x0 * sx, y0 * sy), (x1 - x0) * sx, (y1 - y0) * sy, fill=False, ec=color, lw=2.8)
    )
    handles.append(Line2D([0], [0], color=color, lw=2.8, label=name))
ax.set_axis_off()
ax.set_title(f"{sample}  Hyperplastic AECs on the H&E")
ax.legend(
    handles=handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=9,
    handlelength=1.8,
)
fig.savefig(ROI_DIR / "overview_hyperplastic_aec.pdf", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

# Pathologist regions: concave hull of each annotated instance, legend outside on the right.
ANN_COLOR = {
    "Hyperplastic AECs": "#E7298A",
    "Remodeled Epithelium": "#1F78B4",
    "Epithelial Detachment": "#222222",
    "Large Airway": "#C4A35A",
    "Goblet Cell Metaplasia": "#E67E22",
    "Mixed Inflammation": "#7B2D8E",
    "Muscularized Artery": "#D62728",
    "Fibroblastic Focus": "#2CA02C",
    "Fibrosis": "#8C6D1F",
    "Severe Fibrosis": "#1B9E77",
}
fig_w = 7.2
fig, ax = plt.subplots(
    figsize=(fig_w, fig_w * H / W), dpi=300, facecolor="white"
)
# Map the thumbnail onto the full H&E pixel frame so outlines are not stretched.
ax.imshow(low, extent=(0, W, H, 0), interpolation="bilinear")
boundary_handles = []
for ann_name, color in ANN_COLOR.items():
    sub = cells.loc[cells["Annotation_Type"] == ann_name]
    drew = False
    for _, g in sub.groupby("annotation_type_instance"):
        xy = g[["X_pix_HE", "Y_pix_HE"]].to_numpy(dtype=np.float64)
        if len(xy) < 3:
            continue
        hull = concave_hull(MultiPoint(xy), ratio=0.45, allow_holes=False)
        geoms = [hull] if hull.geom_type == "Polygon" else list(getattr(hull, "geoms", []))
        for geom in geoms:
            if geom.is_empty or geom.geom_type != "Polygon":
                continue
            ring = np.asarray(geom.exterior.coords)
            ax.plot(ring[:, 0], ring[:, 1], color=color, lw=2.8, solid_capstyle="round")
            drew = True
    if drew:
        boundary_handles.append(Line2D([0], [0], color=color, lw=2.8, label=ann_name))
bar = 500 * PX_PER_UM
x_bar = W - bar - 700
y_bar = H - 900
ax.plot([x_bar, x_bar + bar], [y_bar, y_bar], color="black", lw=4, solid_capstyle="butt", zorder=6)
ax.text(x_bar + bar / 2, y_bar - 180, "500 µm", color="black", fontsize=12, ha="center", va="bottom", zorder=6)
ax.set_xlim(0, W)
ax.set_ylim(H, 0)
ax.set_aspect("equal")
ax.set_axis_off()
ax.set_title(f"{sample}  pathologist annotation boundaries")
ax.legend(
    handles=boundary_handles,
    loc="center left",
    bbox_to_anchor=(1.02, 0.5),
    frameon=False,
    fontsize=12,
    handlelength=1.8,
)
ax.set_axis_off()
with plt.rc_context({"pdf.fonttype": 42, "ps.fonttype": 42}):
    fig.savefig(ROI_DIR / "overview_annotation_boundaries.pdf", bbox_inches="tight", facecolor="white")
plt.show()

crops = []
for (name, x0, x1, y0, y1, n), color in zip(boxes, colors):
    crop = np.asarray(he[y0:y1, x0:x1])
    crops.append((name, crop, n, color))
    fig_i = plt.figure(figsize=(crop.shape[1] / 300, crop.shape[0] / 300), dpi=300, facecolor="white")
    ax = fig_i.add_axes([0, 0, 1, 1])
    ax.imshow(crop)
    ax.set_axis_off()
    bar = int(round(100 * PX_PER_UM))
    yb = crop.shape[0] - 28
    ax.plot([16, 16 + bar], [yb, yb], color="white", lw=2, solid_capstyle="butt")
    ax.text(16, yb - 8, "100 um", color="white", fontsize=12, va="bottom")
    fig_i.savefig(ROI_DIR / f"{name}.png", dpi=300, facecolor="white")
    plt.close(fig_i)

fig, axes = plt.subplots(
    1,
    len(crops),
    figsize=(
        sum(c.shape[1] for _, c, _, _ in crops) / 300,
        max(c.shape[0] for _, c, _, _ in crops) / 300,
    ),
    dpi=300,
    facecolor="white",
    gridspec_kw={"width_ratios": [c.shape[1] for _, c, _, _ in crops], "wspace": 0.04},
)
for ax, (name, crop, n, color) in zip(axes, crops):
    ax.imshow(crop)
    ax.set_title(f"{name}\nn={n}", fontsize=20, color=color)
    ax.set_axis_off()
    bar = int(round(100 * PX_PER_UM))
    yb = crop.shape[0] - 28
    ax.plot([16, 16 + bar], [yb, yb], color="white", lw=2, solid_capstyle="butt")
    ax.text(16, yb - 8, "100 um", color="white", fontsize=18, va="bottom")
fig.savefig(ROI_DIR / "hyperplastic_aec_crops.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(ROI_DIR)


In [ ]:
# import tifffile
# import anndata as ad

# # StarDist spatial_HE is already in VUILD107MA-HE.tif pixels (4.705882 px/um).
# sample = "VUILD107MA"
# star = ad.read_h5ad(iv.pred_paths(DATASET, sample)["star"])
# xy = np.asarray(star.obsm["spatial_HE"], dtype=np.float64)
# labels = star.obs["pred_CT"].astype(str).to_numpy()
# palette = iv.PALETTES["L2"]
# DPI = 300
# DOT_PX = 8
# point_s = (DOT_PX * 72 / DPI) ** 2

# for name, x0, x1, y0, y1, n_ann in boxes:
#     crop = np.asarray(he[y0:y1, x0:x1])
#     # fade = (crop.astype(np.float32) * 0.45 + 255 * 0.55).clip(0, 255).astype(np.uint8)
#     fade = (crop.astype(np.float32) * 0.75 + 255 * 0.25).clip(0, 255).astype(np.uint8)
#     inside = (xy[:, 0] >= x0) & (xy[:, 0] < x1) & (xy[:, 1] >= y0) & (xy[:, 1] < y1)
#     hgt, wid = fade.shape[:2]
#     fig = plt.figure(figsize=(wid / DPI, hgt / DPI), dpi=DPI, facecolor="white")
#     ax = fig.add_axes([0, 0, 1, 1])
#     ax.imshow(fade)
#     for cls in palette:
#         m = inside & (labels == cls)
#         if not np.any(m):
#             continue
#         ax.scatter(
#             xy[m, 0] - x0, xy[m, 1] - y0,
#             s=8*point_s, c=[palette[cls][:3]], linewidths=0, zorder=3,
#         )
#     ax.set_xlim(0, wid)
#     ax.set_ylim(hgt, 0)
#     ax.plot([0, wid, wid, 0, 0], [0, 0, hgt, hgt, 0], color="black", lw=1.2, zorder=5)
#     ax.set_axis_off()
#     ax.text(
#         0.02, 0.07, "Hyperplastic AECs", transform=ax.transAxes,
#         fontsize=10, color="black", va="bottom", ha="left", zorder=6,
#         bbox=dict(boxstyle="square,pad=0.28", fc="white", ec="black", lw=0.7),
#     )
#     bar = int(round(20 * PX_PER_UM))
#     ax.plot([14, 14 + bar], [hgt - 16, hgt - 16], color="black", lw=2.5, solid_capstyle="butt", zorder=6)
#     out = ROI_DIR / f"{name}_stardist_pred.png"
#     fig.savefig(out, dpi=DPI, facecolor="white")
#     print(f"{name}: {int(inside.sum())} StarDist nuclei -> {out}")
#     plt.show()


In [ ]:
import importlib
import tifffile
import anndata as ad
import panel_figures
importlib.reload(panel_figures)
from panel_figures import plot_annotated_roi_nuclei

# style="dots" : cell-type markers. style="fill" : StarDist polygons filled by cell type.
PX_PER_UM = 4705882 / 1_000_000
sample = "VUILD107MA"
CASE = (
    REPO
    / "data/Xemium/weiqin/SpatialPF-NGenetics/Spatial-PF-Processed/Data/Complete_Cases"
    / sample
)
ROI_DIR = REPO / "code/Internal_val/figures_vuild107ma"
ROI_DIR.mkdir(parents=True, exist_ok=True)
ann = pd.read_csv(
    CASE / f"{sample}_cells_partitioned_by_annotation_sample_match_with_pixel.csv",
    usecols=["annotation_type_instance", "X_pix_HE", "Y_pix_HE"],
)
ann = ann.loc[ann["annotation_type_instance"].astype(str).str.startswith("hyperplastic_aec")]
he = tifffile.memmap(CASE / f"{sample}-HE.tif")
star = ad.read_h5ad(iv.pred_paths(DATASET, sample)["star"])
plot_annotated_roi_nuclei(
    he,
    ann,
    star.obsm["spatial_HE"],
    star.obs["pred_CT"].astype(str).to_numpy(),
    iv.PALETTES["L2"],
    style="dots",
    px_per_um=PX_PER_UM,
    save_dir=ROI_DIR,
)


In [ ]:
import importlib
import tifffile
import anndata as ad
import panel_figures
importlib.reload(panel_figures)
from panel_figures import plot_annotated_roi_nuclei

# style="fill" : cell-type markers. style="fill" : StarDist polygons filled by cell type.
PX_PER_UM = 4705882 / 1_000_000
sample = "VUILD107MA"
CASE = (
    REPO
    / "data/Xemium/weiqin/SpatialPF-NGenetics/Spatial-PF-Processed/Data/Complete_Cases"
    / sample
)
ROI_DIR = REPO / "code/Internal_val/figures_vuild107ma"
ROI_DIR.mkdir(parents=True, exist_ok=True)
ann = pd.read_csv(
    CASE / f"{sample}_cells_partitioned_by_annotation_sample_match_with_pixel.csv",
    usecols=["annotation_type_instance", "X_pix_HE", "Y_pix_HE"],
)
ann = ann.loc[ann["annotation_type_instance"].astype(str).str.startswith("hyperplastic_aec")]
he = tifffile.memmap(CASE / f"{sample}-HE.tif")
star = ad.read_h5ad(iv.pred_paths(DATASET, sample)["star"])
plot_annotated_roi_nuclei(
    he,
    ann,
    star.obsm["spatial_HE"],
    star.obs["pred_CT"].astype(str).to_numpy(),
    iv.PALETTES["L2"],
    style="fill",
    polys_path=sorted((REPO / "data/Xemium/LUNG/StarDist_Segment" / sample).glob("polys_prob*.pckl"))[-1],
    px_per_um=PX_PER_UM,
    save_dir=ROI_DIR,
)


## VUILD107MA TLS

The TLS crop in Extended Data Fig. 7d is VUILD110LA. This H&E has no TLS polygon. The ROI below is a discrete lymphoid aggregate on this section: a 240 µm window centered near (1274, 2619) µm, then cells within 200 µm (952 B cells, 299 T cells, B fraction 0.71). It sits under ciliated airway epithelium and falls inside the Mixed Inflammation annotation. The crop keeps about 170 µm of that airway above the aggregate. Dots are StarDist nuclei within 20 µm of those B, T, plasma, and DC cells.


In [ ]:
import importlib

import numpy as np
import tifffile
import anndata as ad
import panel_figures

importlib.reload(panel_figures)
from panel_figures import plot_annotated_roi_nuclei

# Discrete B/T aggregate under airway epithelium. Not the published VUILD110LA crop.
# style="dots" draws cell-type markers. style="fill" fills the StarDist polygons.
PX_PER_UM = 4705882 / 1_000_000
DPI = 300
# STYLE = "dots"
STYLE = "fill"
CENTER_UM = (1274.0, 2619.0)
RADIUS_UM = 200.0
TLS_CT = {
    "B cells", "Proliferating B cells",
    "CD4+ T-cells", "CD8+ T-cells", "Tregs", "Proliferating T-cells",
    "Plasma", "Migratory DCs", "cDCs", "pDCs",
}
sample = "VUILD107MA"

CASE = (
    REPO
    / "data/Xemium/weiqin/SpatialPF-NGenetics/Spatial-PF-Processed/Data/Complete_Cases"
    / sample
)
ROI_DIR = REPO / "code/Internal_val/figures_vuild107ma"
ROI_DIR.mkdir(parents=True, exist_ok=True)
cells = pd.read_csv(
    CASE / f"{sample}_cells_partitioned_by_annotation_sample_match_with_pixel.csv",
    usecols=["final_CT", "x_centroid", "y_centroid", "X_pix_HE", "Y_pix_HE"],
)
dist_um = np.hypot(cells["x_centroid"] - CENTER_UM[0], cells["y_centroid"] - CENTER_UM[1])
core = cells.loc[(dist_um < RADIUS_UM) & cells["final_CT"].isin(TLS_CT)].copy()
core["annotation_type_instance"] = "tls_lymphoid_aggregate"
pts_ann = core[["X_pix_HE", "Y_pix_HE"]].to_numpy(dtype=np.float64)
print(
    f"TLS core: {len(core)} cells "
    f"({int(core['final_CT'].isin(['B cells', 'Proliferating B cells']).sum())} B, "
    f"{int(core['final_CT'].str.contains('T-cell|Tregs').sum())} T)"
)

he = tifffile.memmap(CASE / f"{sample}-HE.tif")
H, W = he.shape[:2]
pad_x = int(round(80 * PX_PER_UM))
pad_top = int(round(170 * PX_PER_UM))
pad_bot = int(round(70 * PX_PER_UM))
x0 = max(0, int(np.floor(pts_ann[:, 0].min()) - pad_x))
x1 = min(W, int(np.ceil(pts_ann[:, 0].max()) + pad_x))
y0 = max(0, int(np.floor(pts_ann[:, 1].min()) - pad_top))
y1 = min(H, int(np.ceil(pts_ann[:, 1].max()) + pad_bot))
crop = np.asarray(he[y0:y1, x0:x1])
hgt, wid = crop.shape[:2]

def _frame(ax, image):
    ax.imshow(image)
    ax.set_xlim(0, wid)
    ax.set_ylim(hgt, 0)
    ax.plot([0, wid, wid, 0, 0], [0, 0, hgt, hgt, 0], color="black", lw=1.4, zorder=5)
    ax.set_axis_off()
    ax.text(
        0.02, 0.08, "TLS", transform=ax.transAxes,
        fontsize=26, color="black", va="bottom", ha="left", zorder=6,
        bbox=dict(boxstyle="square,pad=0.3", fc="white", ec="black", lw=2.0),
    )
    bar = int(round(20 * PX_PER_UM))
    ax.plot([16, 16 + bar], [hgt - 18, hgt - 18], color="black", lw=3, solid_capstyle="butt", zorder=6)

fig = plt.figure(figsize=(wid / DPI, hgt / DPI), dpi=DPI, facecolor="white")
ax = fig.add_axes([0, 0, 1, 1])
_frame(ax, crop)
out_he = ROI_DIR / "tls_lymphoid_aggregate.png"
fig.savefig(out_he, dpi=DPI, facecolor="white")
print(out_he.name)
plt.show()


star = ad.read_h5ad(iv.pred_paths(DATASET, sample)["star"])
plot_annotated_roi_nuclei(
    he,
    core,
    star.obsm["spatial_HE"],
    star.obs["pred_CT"].astype(str).to_numpy(),
    iv.PALETTES["L2"],
    style=STYLE,
    polys_path=sorted((REPO / "data/Xemium/LUNG/StarDist_Segment" / sample).glob("polys_prob*.pckl"))[-1],
    px_per_um=PX_PER_UM,
    pad_x_um=80,
    pad_top_um=170,
    pad_bottom_um=70,
    reach_um=20,
    marker_px=28,
    corner_label="TLS",
    save_dir=ROI_DIR,
    save_format="pdf",
)


In [ ]:
import pickle

import numpy as np
from matplotlib.collections import PolyCollection
from scipy.spatial import cKDTree

# Same TLS window as the cell above. StarDist coord[:, 0] is y and coord[:, 1] is x.
# Nuclei within 20 µm of the TLS core stay green. The rest of the window is gray.
star_poly = (
    REPO / "data/Xemium/LUNG/StarDist_Segment" / sample
    / "polys_prob0.01_nms0.3.pckl"
)
with star_poly.open("rb") as handle:
    packed = pickle.load(handle)
coord = packed["coord"]
points = packed["points"]
cy = points[:, 0]
cx = points[:, 1]
inside = (cx >= x0) & (cx < x1) & (cy >= y0) & (cy < y1)
sel_xy = np.column_stack([cx[inside], cy[inside]])
dist, _ = cKDTree(pts_ann).query(sel_xy, distance_upper_bound=20 * PX_PER_UM)
chosen = np.isfinite(dist)
poly_x = coord[inside, 1, :] - x0
poly_y = coord[inside, 0, :] - y0
verts = np.stack([poly_x, poly_y], axis=-1)
verts = np.concatenate([verts, verts[:, :1, :]], axis=1)

fig = plt.figure(figsize=(wid / DPI, hgt / DPI), dpi=DPI, facecolor="white")
ax = fig.add_axes([0, 0, 1, 1])
ax.imshow(crop)
if np.any(~chosen):
    ax.add_collection(PolyCollection(
        verts[~chosen],
        facecolors="none",
        edgecolors="#8C8C8C",
        linewidths=0.45,
        zorder=2,
    ))
if np.any(chosen):
    ax.add_collection(PolyCollection(
        verts[chosen],
        facecolors="none",
        edgecolors="#3DDC00",
        linewidths=0.45,
        zorder=3,
    ))
    ax.scatter(
        sel_xy[chosen, 0] - x0,
        sel_xy[chosen, 1] - y0,
        s=4,
        c="#F0A202",
        linewidths=0,
        zorder=4,
    )
ax.set_xlim(0, wid)
ax.set_ylim(hgt, 0)
ax.plot([0, wid, wid, 0, 0], [0, 0, hgt, hgt, 0], color="black", lw=1.4, zorder=5)
ax.set_axis_off()
ax.text(
    0.02, 0.08, "TLS", transform=ax.transAxes,
    fontsize=26, color="black", va="bottom", ha="left", zorder=6,
    bbox=dict(boxstyle="square,pad=0.3", fc="white", ec="black", lw=2.0),
)
bar = int(round(20 * PX_PER_UM))
ax.plot([16, 16 + bar], [hgt - 18, hgt - 18], color="black", lw=3, solid_capstyle="butt", zorder=6)
out = ROI_DIR / "tls_lymphoid_aggregate_stardist_boundary.png"
fig.savefig(out, dpi=DPI, facecolor="white")
fig.savefig(out.with_suffix(".pdf"), facecolor="white")
print(f"{int(chosen.sum())} selected nuclei, {int((~chosen).sum())} outside the region -> {out.name}")
plt.show()


In [ ]:
from matplotlib.lines import Line2D

# Two-column L2 legend, column order matches the ROI overlays.
names = [
    n for n in iv.PALETTES["L2"]
    if n in {
        "Epithelial", "Alveolar", "Injury_epithelial", "CD4_T",
        "CD8_T", "Tregs", "NK", "B_cell",
        "Plasma", "Macrophage", "Neutrophil", "DC",
        "Fibroblast", "Myofibroblast", "Stromal", "Endothelial",
    }
]
palette = iv.PALETTES["L2"]
handles = [
    Line2D(
        [0], [0], linestyle="none", marker="o", markersize=7,
        markerfacecolor=palette[n][:3], markeredgecolor="none", label=n,
    )
    for n in names
]
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["pdf.use14corefonts"] = True
fig, ax = plt.subplots(figsize=(3.6, 2.05), dpi=300)
ax.set_axis_off()
ax.legend(
    handles=handles,
    loc="center",
    frameon=False,
    ncol=2,
    fontsize=8,
    handlelength=0.8,
    handletextpad=0.45,
    columnspacing=1.1,
    labelspacing=0.35,
    borderaxespad=0.0,
)
out = REPO / "code/Internal_val/figures_vuild107ma/L2_legend"
out.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out.with_suffix(".png"), dpi=300, bbox_inches="tight", facecolor="white")
fig.savefig(out.with_suffix(".pdf"), bbox_inches="tight", facecolor="white")
print(out.with_suffix(".png"))
plt.show()
